# base

> The CSS reset, document styles built from the tokens, icon sizing, and prose

In [ ]:
#| default_exp base

The base layer styles native HTML elements with the tokens, so a page of plain semantic markup is readable without any component. It has four parts: a reset of the browser's default styles; document styles for text, links, code, lists, and tables; icon sizing; and typrose for long-form prose. We register each part's CSS with `reg.rule` in the `reset` or `base` layer, apart from one rule for the `hidden` attribute in the `utilities` layer.

In [ ]:
from fastcore.utils import *
from importlib.resources import files
from answerui.registry import reg
import answerui.tokens  # installs Registry.context

In [ ]:
from fasthtml.common import *
from answerui.demo import demo

Everything in this notebook has a light and a dark rendering, so `schemes` shows the same markup in both, side by side. The page and text colours re-derive inside each panel, because each panel sets its own `color-scheme`:

In [ ]:
def schemes(*fts):
    "The same markup in a light and a dark panel, side by side"
    panels = [Div(*fts, style=f'color-scheme:{s};background:var(--n-3);color:var(--n2);padding:var(--space)') for s in ('light', 'dark')]
    return Div(*panels, style='display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))')

## Reset

The browser's default stylesheet puts margins on headings and paragraphs, gives form controls their own font, and lets images overflow. We undo these once, here. We set block elements' margins to zero with a `:where()` selector. A `:where()` selector has zero specificity, so any later rule wins. Spacing then comes only from a layout container or a prose region, so a parent controls the gaps between its children. Controls inherit the surrounding font and colour, and media is constrained to its container.

In [ ]:
reg.rule(r"""
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
""", layer='reset', doc='Border-box sizing, no block margins, media constrained, controls inheriting text.')

reset [reset]: Border-box sizing, no block margins, media constrained, controls inheriting text.

## Document

The page background is `--n-3`, the base rung, and the text is `--n2`, the top of the neutral ladder. Both colours follow `--hue` and `color-scheme` throughout the document. Headings, links, and code use the type and colour tokens directly. `body` has `container-type: inline-size`, so a component can use a container query to measure the width it actually has, rather than a media query on the viewport. `:root` sets `accent-color` and `caret-color` to `--v`, so native controls follow the hue even before any component styles them.

Native scrollbars have a thumb in the muted rung, `--n1`, and a transparent track. They follow the surrounding palette without changing the browser's scrollbar width or behaviour. The document keeps a gutter for the vertical scrollbar whether or not the page scrolls, so moving between a short page and a long one does not shift the content sideways.

Scrolling to an anchor link's target or to a control that takes focus is smooth, but instant when the user prefers reduced motion. The element with keyboard focus is outlined with `--ring`. A component that shows focus around a larger box, such as a group around its input, outlines that box with the same `--ring`. Pressable controls show the `pointer` cursor, and disabled ones show `not-allowed`.

`mark` is a colour context on the warning hue, so its background is `--v-3` and its text is `--v2`, both from the warning family.

In [ ]:
reg.context('mark', 'var(--hue-warning)')
reg.rule(r"""
:root { accent-color: var(--v); caret-color: var(--v); background: var(--n-3); --ring: 2px solid var(--v-1); }
html { scrollbar-gutter: stable; scroll-behavior: smooth; @media (prefers-reduced-motion: reduce) { scroll-behavior: auto; } }
* { scrollbar-color: var(--n1) transparent; }
body { color: var(--n2); font: var(--weight) var(--text)/var(--leading) var(--font); container-type: inline-size; }
button, a[button], input:is([type=checkbox], [type=radio], [type=range]) { cursor: pointer; }
:disabled { cursor: not-allowed; }
h1, h2, h3, h4, h5, h6 { font-family: var(--font-heading, inherit); font-weight: var(--weight1); line-height: var(--leading-1); text-wrap: balance; }
h1 { font-size: var(--text3); } h2 { font-size: var(--text2); } h3 { font-size: var(--text1); }
p { text-wrap: pretty; }
a { color: var(--v); }
code, kbd, pre, samp { font-family: var(--font-mono); }
code, kbd { font-size: .875em; padding: .1em .3em; border-radius: var(--radius-1); }
code { background: var(--n-2); }
kbd { border: 1px solid var(--n-1); box-shadow: var(--raised); }
pre { background: var(--n-2); padding: var(--space); border-radius: var(--radius); overflow-x: auto;
  code { font-size: inherit; padding: 0; background: none; } }
ul, ol { padding-inline-start: var(--space1); }
blockquote { border-inline-start: 3px solid var(--n-1); padding-inline-start: var(--space); color: var(--n1); }
hr { border: 0; border-block-start: 1px solid var(--n-1); }
small, figcaption { font-size: var(--text-1); }
figcaption { color: var(--n1); }
table {
  border-collapse: separate; border-spacing: 0;
  caption { caption-side: bottom; text-align: start; padding: var(--space-2) var(--space); font-size: var(--text-1); color: var(--n1); }
  :is(th, td) { padding: var(--space-2) var(--space); text-align: start; }
  thead th { font-weight: var(--weight1); border-block-end: 1px solid var(--n); }
  tbody tr + tr > * { border-block-start: 1px solid var(--n-1); }
  tfoot tr > * { border-block-start: 1px solid var(--n); font-weight: var(--weight1); }
  :is(th, td):is([num], [align=right]) { text-align: end; font-variant-numeric: tabular-nums; }
}
:focus-visible { outline: var(--ring); outline-offset: 2px; }
mark { background: var(--v-3); color: var(--v2); padding: .1em .2em; border-radius: var(--radius-1); }
""", layer='base', doc='The page, text, headings, links, code, lists, quotes, tables, rules, marks, focus, and cursors on the tokens.')

document [base]: The page, text, headings, links, code, lists, quotes, tables, rules, marks, focus, and cursors on the tokens.

The `hidden` attribute needs a page-wide rule because any `display` that a component sets overrides the browser's built-in rule for `hidden`. Without our rule, a hidden card or alert would stay on screen. Our rule is in the `utilities` layer, which comes after the `components` and `app` layers, so the attribute wins over any `display` set there. It skips `hidden=until-found`, because the browser reveals that content itself when find-in-page matches text in it or a fragment link targets it:

In [ ]:
reg.rule('[hidden]:not([hidden=until-found]) { display: none; }', layer='utilities', doc='The hidden attribute wins over any display a component sets.')

hidden [utilities]: The hidden attribute wins over any display a component sets.

This sample holds only content elements. None of them has a margin, so the sample spaces them with a grid gap, as a layout container would:

In [ ]:
sample = Div(
    H1('A document on the ladder'),
    P('Body text with ', A('a link', href='#'), ', ', Code('code'), ', ', Kbd('Ctrl'), ' and a ', Mark('marked'), ' phrase.'),
    H2('Lists, quotes, and tables'),
    Ul(Li('Bullets take the hairline rung'), Li('Captions and quotes take the muted rung')),
    Blockquote('Quoted text sits on a hairline and drops to the muted rung.'),
    Pre(Code('print("code blocks sit on the raised surface")')),
    Table(Thead(Tr(Th('Rung'), Th('Job'))), Tbody(Tr(Td('n2'), Td('text')), Tr(Td('v'), Td('links')))),
    Hr(), Small('Small print.'),
    style='display:grid;gap:var(--space)')
demo(schemes(sample))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div style="display:grid;gap:var(--space)">
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </div>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div style="display:grid;gap:var(--space)">
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </div>
  </div>
</div>

```

Headings use the same font as body text. To give the headings in part of a page a different font, set `--font-heading` there. The second article here uses `--font-serif`:


In [ ]:
demo(Article(H2('Account settings'), P('This heading is in the body face.')),
    Article(H2('A short history of type'), P('This one is in the serif, set by its article.'), style='--font-heading: var(--font-serif)'))


```html
<article>
  <h2>Account settings</h2>
  <p>This heading is in the body face.</p>
</article>
<article style="--font-heading: var(--font-serif)">
  <h2>A short history of type</h2>
  <p>This one is in the serif, set by its article.</p>
</article>

```

## Icons

Icons are small inline SVGs in text, links, and buttons. answerui takes them from [fastlucide](https://github.com/AnswerDotAI/fastlucide), the Lucide set for FastHTML. The base layer only sizes and aligns each icon to fit wherever it is placed.

A sprite icon is an `svg` that contains only a `use` element. The `use` points at a `symbol` in a hidden sheet elsewhere in the page, and the symbol holds the shape. fastlucide, like most icon sets, works this way, so each shape is defined once however many times the page uses it. To place an icon, import its constructor from `fastlucide.icons` and call it. `reg.ftrs()` adds the sheet to the end of every page, and `demo` adds it to every preview. The icon rule matches any `svg` with a `use` child, so a sprite icon needs no attribute of ours. Mark an SVG drawn in full, such as a brand logo, with the `icon` attribute instead.

fastlucide's icons, and any SVG that paints with `currentColor`, take the colour of the surrounding text. We size icons at `1.25em`, so their size follows the font size rather than the `width` and `height` in their markup. An icon in a heading, a button, or a caption therefore fits its line without being given a size. Setting `flex: none` stops a cluster or a button from squeezing the icon when space is short.

In [ ]:
reg.rule(r'''
svg:is(:has(> use), [icon]) { inline-size: 1.25em; block-size: 1.25em; vertical-align: -.3em; flex: none; }
''', layer='base', doc='A sprite icon, or an inline SVG marked `icon`, sized and aligned to the text around it.')

icons [base]: A sprite icon, or an inline SVG marked `icon`, sized and aligned to the text around it.

Here fastlucide's `X` appears in a heading, a paragraph, and small print. The link holds a filled square marked `icon`:

In [ ]:
from fasthtml.svg import Path
from fastlucide.icons import X

In [ ]:
square = Svg(Path(d='M4 4h16v16H4z'), viewBox='0 0 24 24', fill='currentColor', icon=True)
demo(Div(H2(X(), ' A heading'), P(X(), ' Body text with ', A(square, ' a link', href='#'), '.'), Small(X(), ' Small print'), style='display:grid;gap:var(--space)'))

```html
<div style="display:grid;gap:var(--space)">
  <h2>
<svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> A heading  </h2>
  <p>
<svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> Body text with <a href="#"><svg viewbox="0 0 24 24" fill="currentColor" icon><path d="M4 4h16v16H4z"></path></svg> a link</a>.  </p>
<small><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-x"></use></svg> Small print</small></div>

```

## Prose

Long-form content needs more than the styles for controls and layout: rhythm between blocks, a line length that suits reading, and typographic care in lists, quotes, and code. [typrose](https://github.com/AnswerDotAI/typrose) provides these. It is a CSS-native port of Tailwind's typography plugin. Its rules are scoped under a `prose` class, and every colour it uses is a `--prose-*` property set on that element.

We bundle typrose's stylesheet and rewrite its selectors when we register it, because answerui marks scopes with attributes rather than classes. `.prose` becomes `[prose]`, to match `Article(prose=True)`. A size variant such as `.prose-lg` becomes `[prose="lg"]`, to match `Article(prose='lg')`. The value is quoted because an unquoted value such as `2xl` is not valid CSS. The same step removes typrose's grey theme classes and its dark-mode `invert` block, because the ladder provides colour themes and dark mode:

In [ ]:
TYPROSE = files('answerui').joinpath('css/typrose.css').read_text()
TYPROSE = re.sub(r'(/\*[^*]*\*/\n)?\.prose-(slate|zinc|neutral|stone|invert) \{[^}]*\}\n*', '', TYPROSE)
TYPROSE = re.sub(r'\.prose-(\w+)', r'[prose="\1"]', TYPROSE)
TYPROSE = re.sub(r'\.prose\b', '[prose]', TYPROSE)
reg.rule(TYPROSE, layer='base', doc='Prose typography from typrose, on the `prose` attribute.')

typrose [base]: Prose typography from typrose, on the `prose` attribute.

typrose's own colours are a fixed grey scale. In a later rule in the same layer, we set each typrose colour property on `[prose]` to a rung, so prose follows `--hue`, `color-scheme`, and any colour context it sits in. We leave two colours fixed: the `hr` divider icon is an SVG image with a fixed stroke, and the `kbd` shadow is black at 10% opacity. The same rule caps a prose region at `65ch` wide, a limit typrose leaves to the page. It also sets the font size to `--text` times `--prose-font-scale`, whereas typrose multiplies `--prose-font-scale` by a fixed `1rem`. Prose keeps typrose's rhythm, tables included. `--prose-scale` sets prose spacing relative to the prose font size, and `--space` sets the spacing of controls and layout. `--text` is the base font size for both:

In [ ]:
reg.rule(r"""
[prose] {
  max-inline-size: 65ch; font-size: calc(var(--text) * var(--prose-font-scale));
  --prose-body: var(--n2); --prose-headings: var(--n2); --prose-bold: var(--n2); --prose-quotes: var(--n2); --prose-code: var(--n2);
  --prose-links: var(--v);
  --prose-lead: var(--n1); --prose-counters: var(--n1); --prose-captions: var(--n1);
  --prose-bullets: var(--n-1); --prose-hr: var(--n-1); --prose-quote-borders: var(--n-1);
  --prose-th-borders: var(--n); --prose-td-borders: var(--n-1);
}
""", layer='base', doc='typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.')

prose [base]: typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.

The same content as a `prose` article, with typrose's rhythm and line length and the ladder's colours:

In [ ]:
article = Article(*sample.children, prose=True)
demo(schemes(article))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <article prose>
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </article>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <article prose>
      <h1>A document on the ladder</h1>
      <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.      </p>
      <h2>Lists, quotes, and tables</h2>
      <ul>
        <li>Bullets take the hairline rung</li>
        <li>Captions and quotes take the muted rung</li>
      </ul>
      <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>      <table>
        <thead>
          <tr>
            <th>Rung</th>
            <th>Job</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>n2</td>
            <td>text</td>
          </tr>
          <tr>
            <td>v</td>
            <td>links</td>
          </tr>
        </tbody>
      </table>
<hr><small>Small print.</small>    </article>
  </div>
</div>

```

Inside a `danger` context, every colour in the article re-derives from the danger hue, because each one is a rung:

In [ ]:
demo(schemes(Div(article, danger=True)))

```html
<div style="display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))">
  <div style="color-scheme:light;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div danger>
      <article prose>
        <h1>A document on the ladder</h1>
        <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.        </p>
        <h2>Lists, quotes, and tables</h2>
        <ul>
          <li>Bullets take the hairline rung</li>
          <li>Captions and quotes take the muted rung</li>
        </ul>
        <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>        <table>
          <thead>
            <tr>
              <th>Rung</th>
              <th>Job</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td>n2</td>
              <td>text</td>
            </tr>
            <tr>
              <td>v</td>
              <td>links</td>
            </tr>
          </tbody>
        </table>
<hr><small>Small print.</small>      </article>
    </div>
  </div>
  <div style="color-scheme:dark;background:var(--n-3);color:var(--n2);padding:var(--space)">
    <div danger>
      <article prose>
        <h1>A document on the ladder</h1>
        <p>
Body text with <a href="#">a link</a>, <code>code</code>, <kbd>Ctrl</kbd> and a <mark>marked</mark> phrase.        </p>
        <h2>Lists, quotes, and tables</h2>
        <ul>
          <li>Bullets take the hairline rung</li>
          <li>Captions and quotes take the muted rung</li>
        </ul>
        <blockquote>Quoted text sits on a hairline and drops to the muted rung.</blockquote>
<pre><code>print("code blocks sit on the raised surface")</code></pre>        <table>
          <thead>
            <tr>
              <th>Rung</th>
              <th>Job</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td>n2</td>
              <td>text</td>
            </tr>
            <tr>
              <td>v</td>
              <td>links</td>
            </tr>
          </tbody>
        </table>
<hr><small>Small print.</small>      </article>
    </div>
  </div>
</div>

```

Headings, native controls, and prose all size from the same base tokens. The compact region below sets `--space` and `--text`, and uses `--text` as its own font size. Its heading, control, and prose shrink with it, and the prose keeps its own rhythm:

In [ ]:
def region(title, style=''):
    "A heading, a paragraph, a native control, and a prose region"
    return Div(H2(title), P('Body text at the region\'s size.'), Input(placeholder='A native control'),
        Article(P('Prose sets its size from the same base and keeps its own rhythm.'), P('A second paragraph shows that rhythm.'), prose=True),
        style=f'font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);{style}')

demo(Div(region('Normal'), region('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

```html
<div style="display:grid;grid-template-columns:1fr 1fr">
  <div style="font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);">
    <h2>Normal</h2>
    <p>Body text at the region's size.</p>
    <input placeholder="A native control">
    <article prose>
      <p>Prose sets its size from the same base and keeps its own rhythm.</p>
      <p>A second paragraph shows that rhythm.</p>
    </article>
  </div>
  <div style="font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);--space:.75rem;--text:.875rem">
    <h2>Compact</h2>
    <p>Body text at the region's size.</p>
    <input placeholder="A native control">
    <article prose>
      <p>Prose sets its size from the same base and keeps its own rhythm.</p>
      <p>A second paragraph shows that rhythm.</p>
    </article>
  </div>
</div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()